In [8]:
!pip install "numpy<2" matplotlib --upgrade --quiet


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [1]:
!pip install tensorflow opencv-python numpy matplotlib scikit-learn seaborn --quiet


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [3]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print('TensorFlow version:', tf.__version__)
print(np.__version__)
print('All libraries imported successfully! ✅')

TensorFlow version: 2.21.0
2.2.6
All libraries imported successfully! ✅


In [6]:
import os
print(os.getcwd())

/home/rgukt


In [1]:
import os
print(os.listdir('/home/rgukt/dataset/train'))

['rottenoranges', 'freshapples', 'freshoranges', 'rottenapples', 'rottenbanana', 'freshbanana']


In [4]:
DATASET_PATH = '/home/rgukt/dataset'

TRAIN_DIR   = os.path.join(DATASET_PATH, 'train')
TEST_DIR    = os.path.join(DATASET_PATH, 'test')
IMG_SIZE    = (100, 100)
BATCH_SIZE  = 32
EPOCHS      = 20
NUM_CLASSES = 6

# Verify dataset exists
print('Train folder exists:', os.path.exists(TRAIN_DIR))
print('Test  folder exists:', os.path.exists(TEST_DIR))
print('Classes found:', os.listdir(TRAIN_DIR))


Train folder exists: True
Test  folder exists: True
Classes found: ['rottenoranges', 'freshapples', 'freshoranges', 'rottenapples', 'rottenbanana', 'freshbanana']


In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

DATASET_PATH = '/home/rgukt/dataset'
TRAIN_DIR   = os.path.join(DATASET_PATH, 'train')
TEST_DIR    = os.path.join(DATASET_PATH, 'test')
IMG_SIZE    = (100, 100)
BATCH_SIZE  = 32
EPOCHS      = 20
NUM_CLASSES = 6

print('All set! ✅')

/home/rgukt/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
I0000 00:00:1781509384.401083   90502 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1781509391.821257   90502 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781509412.370798   90502 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


All set! ✅


In [4]:
train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    validation_split=0.2
)

test_datagen = ImageDataGenerator(rescale=1.0/255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training', shuffle=True
)

val_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation', shuffle=False
)

test_gen = test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

class_names = list(train_gen.class_indices.keys())
print('Classes:', class_names)

Found 8723 images belonging to 6 classes.
Found 2178 images belonging to 6 classes.
Found 2698 images belonging to 6 classes.
Classes: ['freshapples', 'freshbanana', 'freshoranges', 'rottenapples', 'rottenbanana', 'rottenoranges']


In [3]:
# Show one batch of images to verify loading
images, labels = next(train_gen)

plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(images[i])
    plt.title(class_names[np.argmax(labels[i])], fontsize=8)
    plt.axis('off')
plt.suptitle('Sample Training Images', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

NameError: name 'train_gen' is not defined

In [5]:
print(type(train_gen))
print(train_gen)

<class 'keras.src.legacy.preprocessing.image.DirectoryIterator'>


In [6]:
images, labels = next(train_gen)
print('Images shape:', images.shape)
print('Labels shape:', labels.shape)
print('✅ Data loading works!')

Images shape: (32, 100, 100, 3)
Labels shape: (32, 6)
✅ Data loading works!


In [7]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

images, labels = next(train_gen)

plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(images[i])
    plt.title(class_names[np.argmax(labels[i])], fontsize=8)
    plt.axis('off')
plt.suptitle('Sample Training Images', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/home/rgukt/sample_images.png', dpi=150)
plt.close()
print('✅ Image saved! Check /home/rgukt/sample_images.png')

✅ Image saved! Check /home/rgukt/sample_images.png


In [8]:
model = models.Sequential([

    # Block 1
    layers.Conv2D(32, (3,3), activation='relu', padding='same',
                  input_shape=(100, 100, 3)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.25),

    # Block 2
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.25),

    # Block 3
    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.4),

    # Classifier
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(6, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()
print('✅ Model built successfully!')

/home/rgukt/.local/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1781509982.459220   90502 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 100, 100, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 100, 100, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 100, 100, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 50, 50, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 50, 50, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 50, 50, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 50, 50, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 50, 50, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 25, 25, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 25, 25, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 25, 25, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 25, 25, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 25, 25, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 18432)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     4,718,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,009,318 (19.11 MB)

 Trainable params: 5,008,358 (19.11 MB)

 Non-trainable params: 960 (3.75 KB)

✅ Model built successfully!


In [9]:
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint('/home/rgukt/best_fruit_model.h5', 
                    monitor='val_accuracy',
                    save_best_only=True, verbose=1)
]

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)

print('✅ Training complete!')

Epoch 1/20


I0000 00:00:1781510150.466662   90502 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
W0000 00:00:1781510157.359429   91823 cpu_allocator_impl.cc:82] Allocation of 40960000 exceeds 10% of free system memory.
W0000 00:00:1781510159.190408   91823 cpu_allocator_impl.cc:82] Allocation of 40960000 exceeds 10% of free system memory.
W0000 00:00:1781510159.256105   91824 cpu_allocator_impl.cc:82] Allocation of 40960000 exceeds 10% of free system memory.
W0000 00:00:1781510159.280732   91823 cpu_allocator_impl.cc:82] Allocation of 40960000 exceeds 10% of free system memory.
W0000 00:00:1781510160.968512   91824 cpu_allocator_impl.cc:82] Allocation of 25920000 exceeds 10% of free system memory.



Epoch 1: val_accuracy improved from None to 0.39945, saving model to /home/rgukt/best_fruit_model.h5
.0773   

273/273 ━━━━━━━━━━━━━━━━━━━━ 772s 3s/step - accuracy: 0.7337 - loss: 0.7900 - val_accuracy: 0.3994 - val_loss: 1.9865
Epoch 2/20

Epoch 2: val_accuracy improved from 0.39945 to 0.63774, saving model to /home/rgukt/best_fruit_model.h5
14   

273/273 ━━━━━━━━━━━━━━━━━━━━ 657s 2s/step - accuracy: 0.8231 - loss: 0.5059 - val_accuracy: 0.6377 - val_loss: 1.6429
Epoch 3/20

Epoch 3: val_accuracy did not improve from 0.63774
273/273 ━━━━━━━━━━━━━━━━━━━━ 680s 2s/step - accuracy: 0.8603 - loss: 0.3913 - val_accuracy: 0.6010 - val_loss: 2.0818
Epoch 4/20

Epoch 4: val_accuracy did not improve from 0.63774
273/273 ━━━━━━━━━━━━━━━━━━━━ 656s 2s/step - accuracy: 0.8788 - loss: 0.3386 - val_accuracy: 0.5487 - val_loss: 1.5340
Epoch 5/20

Epoch 5: val_accuracy did not improve from 0.63774
273/273 ━━━━━━━━━━━━━━━━━━━━ 648s 2s/step - accuracy: 0.8952 - loss: 0.2984 - val_accuracy: 0.6166 - val_loss: 1.3487
Epoch 6/20

Epoch 6: val_accuracy did not improve from 0.63774
273/273 ━━━━━━━━━━━━━━━━━━━━ 647s 2s/step - accuracy: 0.9073 - loss: 0.2661 - val_accuracy: 0.5248 - val_loss: 1.8484
Epoch 7/20

Epoch 7: val_accuracy improved from 0.63774 to 0.72819, saving model to /home/rgukt/best_fruit_model.h5
72  

273/273 ━━━━━━━━━━━━━━━━━━━━ 648s 2s/step - accuracy: 0.9065 - loss: 0.2623 - val_accuracy: 0.7282 - val_loss: 0.8946
Epoch 8/20

Epoch 8: val_accuracy improved from 0.72819 to 0.85767, saving model to /home/rgukt/best_fruit_model.h5
62  

273/273 ━━━━━━━━━━━━━━━━━━━━ 650s 2s/step - accuracy: 0.9170 - loss: 0.2369 - val_accuracy: 0.8577 - val_loss: 0.3647
Epoch 9/20

Epoch 9: val_accuracy did not improve from 0.85767
273/273 ━━━━━━━━━━━━━━━━━━━━ 646s 2s/step - accuracy: 0.9231 - loss: 0.2276 - val_accuracy: 0.8402 - val_loss: 0.4355
Epoch 10/20

Epoch 10: val_accuracy did not improve from 0.85767
273/273 ━━━━━━━━━━━━━━━━━━━━ 734s 3s/step - accuracy: 0.9282 - loss: 0.2001 - val_accuracy: 0.7057 - val_loss: 1.0047
Epoch 11/20

Epoch 11: val_accuracy did not improve from 0.85767
273/273 ━━━━━━━━━━━━━━━━━━━━ 729s 3s/step - accuracy: 0.9368 - loss: 0.1778 - val_accuracy: 0.7975 - val_loss: 0.6373
Epoch 12/20

Epoch 12: val_accuracy did not improve from 0.85767
273/273 ━━━━━━━━━━━━━━━━━━━━ 742s 3s/step - accuracy: 0.9327 - loss: 0.1889 - val_accuracy: 0.7383 - val_loss: 0.7402
Epoch 13/20

Epoch 13: val_accuracy did not improve from 0.85767
273/273 ━━━━━━━━━━━━━━━━━━━━ 751s 3s/step - accuracy: 0.9353 - loss: 0.1811 - val_accur

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training History — Fruit Quality Detection CNN', fontsize=13, fontweight='bold')

axes[0].plot(history.history['accuracy'],     label='Train', color='#1A6B9A', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val',   color='#FF9900', linewidth=2)
axes[0].set_title('Accuracy'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'],     label='Train', color='#1A6B9A', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val',   color='#FF9900', linewidth=2)
axes[1].set_title('Loss'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/home/rgukt/training_history.png', dpi=150)
plt.close()
print('✅ Saved to /home/rgukt/training_history.png')

✅ Saved to /home/rgukt/training_history.png


In [11]:
loss, accuracy = model.evaluate(test_gen, verbose=0)
print(f'Test Accuracy : {accuracy*100:.2f}%')
print(f'Test Loss     : {loss:.4f}')

y_pred   = np.argmax(model.predict(test_gen, verbose=0), axis=1)
y_true   = test_gen.classes

print('\n── Classification Report ──')
print(classification_report(y_true, y_pred, target_names=class_names))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix — Fruit Quality Detection')
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('/home/rgukt/confusion_matrix.png', dpi=150)
plt.close()
print('✅ Saved to /home/rgukt/confusion_matrix.png')

Test Accuracy : 85.06%
Test Loss     : 0.4288

── Classification Report ──
               precision    recall  f1-score   support

  freshapples       0.91      0.96      0.94       395
  freshbanana       1.00      0.99      0.99       381
 freshoranges       1.00      0.29      0.45       388
 rottenapples       0.72      0.92      0.81       601
 rottenbanana       0.99      0.99      0.99       530
rottenoranges       0.70      0.87      0.78       403

     accuracy                           0.85      2698
    macro avg       0.89      0.84      0.83      2698
 weighted avg       0.88      0.85      0.83      2698

✅ Saved to /home/rgukt/confusion_matrix.png


In [19]:
def predict_image(image_path):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, IMG_SIZE)
    img_array  = np.expand_dims(img_resized / 255.0, axis=0)

    prediction = model.predict(img_array, verbose=0)
    class_idx  = np.argmax(prediction)
    confidence = prediction[0][class_idx] * 100
    label      = class_names[class_idx]

    color = 'green' if 'fresh' in label else 'red'
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.title(f'{label.upper()}\n{confidence:.1f}% confidence',
              color=color, fontsize=12, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    return label, confidence

# ⚠️ Change this to any fruit image path on your PC
# predict_image('my_apple.jpg')
print('predict_image() is ready — pass any image path to test!')

predict_image() is ready — pass any image path to test!


In [21]:
# Define and call together
def predict_fruit(image_path):
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (100, 100))
    img_array = np.expand_dims(img_resized / 255.0, axis=0)

    prediction = model.predict(img_array, verbose=0)
    class_idx  = np.argmax(prediction)
    confidence = prediction[0][class_idx] * 100
    label      = class_names[class_idx]

    print('\n── Prediction Results ──')
    print(f'Prediction: {label.upper()}')
    print(f'Confidence: {confidence:.2f}%')
    print(f'\n── All Class Probabilities ──')
    for i, name in enumerate(class_names):
        bar = '█' * int(prediction[0][i] * 40)
        print(f'{name:<15} {prediction[0][i]*100:5.2f}% {bar}')

    plt.figure(figsize=(5, 5))
    plt.imshow(img_rgb)
    plt.title(f'{label.upper()}\n{confidence:.1f}% confidence',
              color='green' if 'fresh' in label else 'red',
              fontsize=13, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.savefig('/home/rgukt/prediction_result.png', dpi=150)
    plt.close()
    print('✅ Saved to /home/rgukt/prediction_result.png')

# Call immediately
predict_fruit('/home/rgukt/apple.jpg')


── Prediction Results ──
Prediction: ROTTENORANGES
Confidence: 79.18%

── All Class Probabilities ──
freshapples      0.38% 
freshbanana      0.00% 
freshoranges    16.48% ██████
rottenapples     3.96% █
rottenbanana     0.00% 
rottenoranges   79.18% ███████████████████████████████
✅ Saved to /home/rgukt/prediction_result.png


In [22]:
# Check what the image looks like
img = cv2.imread('/home/rgukt/apple.jpg')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(4,4))
plt.imshow(img_rgb)
plt.title('Your Input Image')
plt.axis('off')
plt.savefig('/home/rgukt/input_check.png', dpi=150)
plt.close()
print('✅ Saved to /home/rgukt/input_check.png')

✅ Saved to /home/rgukt/input_check.png


In [30]:
predict_fruit('/home/rgukt/apple.jpg')


── Prediction Results ──
Prediction: ROTTENORANGES
Confidence: 79.18%

── All Class Probabilities ──
freshapples      0.38% 
freshbanana      0.00% 
freshoranges    16.48% ██████
rottenapples     3.96% █
rottenbanana     0.00% 
rottenoranges   79.18% ███████████████████████████████
✅ Saved to /home/rgukt/prediction_result.png


In [31]:
predict_fruit('/home/rgukt/apple2.jpg')


── Prediction Results ──
Prediction: FRESHAPPLES
Confidence: 76.96%

── All Class Probabilities ──
freshapples     76.96% ██████████████████████████████
freshbanana      1.97% 
freshoranges     4.66% █
rottenapples     3.43% █
rottenbanana     1.68% 
rottenoranges   11.29% ████
✅ Saved to /home/rgukt/prediction_result.png


In [32]:
!pip install streamlit --quiet


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip
